In [1]:
# Added for the public reproduction package: load the pseudonymized CSV instead of the
# private spreadsheet and write any output files to ../output/notebooks/. See repro_data.py.
import repro_data


In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# CONFIGURACIÓN
# --------------------------------------------------------------------------
ARCHIVO = "Wallet-pattern-.xlsx"
HOJA = "Datos cuanti"

COL_TRATAMIENTO = "Grupo"
COL_VALOR = "SUS_Seguridad"

# --------------------------------------------------------------------------
# PARTICIPANTE A EXCLUIR
# --------------------------------------------------------------------------
# Fila de Excel a excluir (1-based, contando el encabezado como fila 1)
EXCEL_ROW_TO_EXCLUDE = 54

# Conversión a índice de pandas:
# Excel fila 2 -> pandas índice 0
PANDAS_IDX_TO_EXCLUDE = EXCEL_ROW_TO_EXCLUDE - 2

# Ítems de seguridad, usados para detectar quién respondió algo del bloque
ITEMS_SEGURIDAD = [
    "Siento que este método de envío de criptoactivos es lo suficientemente "
    "seguro para valores que representen hasta 1 (un) ingreso mensual",

    "Siento que este método de envío de criptoactivos es no lo suficientemente "
    "seguro incluso para valores que representen hasta 2 (dos) ingresos "
    "mensuales",

    "Siento que este método de envío de criptoactivos es lo suficientemente "
    "seguro para la totalidad del monto de todos mis criptoactivos con este "
    "método.",

    "Siento que este método de envío de criptoactivos no es lo suficientemente "
    "seguro para todos mis criptoactivos con este método.",

    "Siento que este método de envío de criptoactivos es lo suficientemente "
    "seguro independiente del monto que representan.",

    "Siento que este método de envío de criptoactivos no es lo suficientemente "
    "seguro independiente del monto que representan.",
]

COL_SEGURIDAD_SUS_RAW = "seguridad sus"

NOMBRES_TRATAMIENTOS = {
    1: "T1 - Mobile",
    2: "T2 - Two devices initiating PC",
    3: "T3 - 2 devices initiating Mobile",
    4: "T4 - PC",
}

# Grupos a graficar:
# 4 tratamientos individuales + Single-sign + Multi-sign
GRUPOS_BOXPLOT = [
    ("T1 - Mobile", [1]),
    ("T2 - Two devices initiating PC", [2]),
    ("T3 - 2 devices initiating Mobile", [3]),
    ("T4 - PC", [4]),
    ("T1 + T4 - One device only", [1, 4]),
    ("T2 + T3 - Two devices", [2, 3]),
]

TITULO_GRAFICO = "SUS Scores Distribution for security per Treatment"
VALOR_REFERENCIA = 36.0


# --------------------------------------------------------------------------
# CARGA DE DATOS
# --------------------------------------------------------------------------
def parse_likert(valor):
    """Convierte valores tipo '5 (Totalmente de acuerdo)' o 3 a float."""
    if pd.isna(valor):
        return np.nan

    if isinstance(valor, (int, float)):
        return float(valor)

    m = re.match(r"^\s*(\d+)", str(valor))

    return float(m.group(1)) if m else np.nan


def cargar_datos(archivo, hoja):

    df = repro_data.read_datos_cuanti()

    print(f"Participantes originales: {len(df)}")

    # ----------------------------------------------------------------------
    # Convertir los seis ítems de seguridad a valores numéricos
    # ----------------------------------------------------------------------
    for col in ITEMS_SEGURIDAD:
        df[col] = df[col].apply(parse_likert)

    # Eliminar filas sin tratamiento
    df = df.dropna(
        subset=[COL_TRATAMIENTO]
    ).copy()

    df[COL_TRATAMIENTO] = (
        df[COL_TRATAMIENTO].astype(int)
    )

    # ----------------------------------------------------------------------
    # FILTRO 1:
    # excluir quienes no respondieron NINGÚN ítem de seguridad
    # ----------------------------------------------------------------------
    respondieron_algo = (
        df[ITEMS_SEGURIDAD]
        .notna()
        .any(axis=1)
    )

    df = df[respondieron_algo].copy()

    print(
        f"Después de excluir quienes no respondieron "
        f"ningún ítem de seguridad: n={len(df)}"
    )

    # ----------------------------------------------------------------------
    # FILTRO 2:
    # excluir participante de la fila 54 de Excel
    # ----------------------------------------------------------------------

    # Verificar que el índice todavía existe
    if PANDAS_IDX_TO_EXCLUDE not in df.index:

        raise ValueError(
            f"No se encontró el índice pandas "
            f"{PANDAS_IDX_TO_EXCLUDE}, correspondiente a "
            f"la fila Excel {EXCEL_ROW_TO_EXCLUDE}."
        )

    fila_excluida = df.loc[PANDAS_IDX_TO_EXCLUDE]

    # Verificar que efectivamente no respondió el primer ítem
    assert pd.isna(
        fila_excluida[ITEMS_SEGURIDAD[0]]
    ), (
        f"La fila Excel {EXCEL_ROW_TO_EXCLUDE} "
        f"(índice pandas {PANDAS_IDX_TO_EXCLUDE}) "
        "NO tiene NaN en el primer ítem de seguridad. "
        "Revisar el mapeo de filas."
    )

    print(
        f"Excluyendo participante de la fila Excel "
        f"{EXCEL_ROW_TO_EXCLUDE} "
        f"(índice pandas {PANDAS_IDX_TO_EXCLUDE})"
    )

    print(
        f"  Grupo: {int(fila_excluida[COL_TRATAMIENTO])}"
    )

    print(
        f"  seguridad sus original: "
        f"{fila_excluida[COL_SEGURIDAD_SUS_RAW]}"
    )

    print(
        "  Motivo: no respondió el primer ítem de seguridad."
    )

    df = df.drop(
        index=PANDAS_IDX_TO_EXCLUDE
    ).copy()

    print(
        f"Después de excluir fila Excel "
        f"{EXCEL_ROW_TO_EXCLUDE}: n={len(df)}"
    )

    # ----------------------------------------------------------------------
    # CALCULAR EL PUNTAJE DE SEGURIDAD
    # ----------------------------------------------------------------------
    #
    # La columna original "seguridad sus" tiene valores entre -9 y 12.
    #
    # Transformación:
    #
    #     score = (seguridad sus + 12) * 2.5
    #
    # Esto lleva el puntaje a la escala 0-60.
    # ----------------------------------------------------------------------

    df[COL_VALOR] = (
        df[COL_SEGURIDAD_SUS_RAW] + 12
    ) * 2.5

    return df


# --------------------------------------------------------------------------
# ESTADÍSTICOS DESCRIPTIVOS
# --------------------------------------------------------------------------
def calcular_estadisticos(
    df,
    col_tratamiento,
    col_valor,
    grupos
):

    """Calcula N, media, mediana, DE, mín y máx."""

    filas = []

    for etiqueta, nums_tratamiento in grupos:

        valores = df.loc[
            df[col_tratamiento].isin(nums_tratamiento),
            col_valor
        ]

        filas.append(
            {
                "Tratamiento": etiqueta,
                "N": len(valores),
                "Media": round(valores.mean(), 2),
                "Mediana": round(valores.median(), 2),
                "Desvío estándar": round(
                    valores.std(ddof=1),
                    2
                ),
                "Mínimo": round(
                    valores.min(),
                    2
                ),
                "Máximo": round(
                    valores.max(),
                    2
                ),
            }
        )

    return pd.DataFrame(filas)


# --------------------------------------------------------------------------
# EXPORTAR LATEX
# --------------------------------------------------------------------------
def exportar_latex(
    resumen,
    archivo="resumen_tratamientos_seguridad.tex"
):

    latex = resumen.to_latex(
        index=False,
        float_format="%.2f",
        caption=(
            "Estadísticos descriptivos del puntaje "
            "SUS de seguridad por tratamiento"
        ),
        label="tab:sus_seguridad_stats",
        column_format="lrrrrrr",
    )

    with open(
        archivo,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(latex)

    print(
        f"Guardado como {archivo}"
    )


# --------------------------------------------------------------------------
# BOXPLOT
# --------------------------------------------------------------------------
def graficar_boxplot(
    df,
    col_tratamiento,
    col_valor,
    grupos,
    titulo,
    valor_referencia,
    archivo_salida="boxplot_tratamientos.png",
    archivo_salida_pdf="boxplot_tratamientos.pdf"
):

    """
    Genera un boxplot con:
      - caja de color
      - puntos individuales con jitter
      - media marcada
      - valor de la media anotado
      - línea de referencia
    """

    df = df[
        [
            col_tratamiento,
            col_valor
        ]
    ].rename(
        columns={
            col_tratamiento: "tratamiento",
            col_valor: "valor"
        }
    )

    df["valor"] = pd.to_numeric(
        df["valor"],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            "tratamiento",
            "valor"
        ]
    )

    df["tratamiento"] = (
        df["tratamiento"].astype(int)
    )

    # ----------------------------------------------------------------------
    # Construir datos de los seis grupos
    # ----------------------------------------------------------------------

    datos_grupos = []
    etiquetas = []

    for etiqueta, nums_tratamiento in grupos:

        valores = df.loc[
            df["tratamiento"].isin(
                nums_tratamiento
            ),
            "valor"
        ].values

        datos_grupos.append(
            valores
        )

        etiquetas.append(
            etiqueta
        )

    n_grupos = len(
        datos_grupos
    )

    # ----------------------------------------------------------------------
    # Colores
    # ----------------------------------------------------------------------

    colores = plt.cm.viridis(
        np.linspace(
            0.15,
            0.85,
            n_grupos
        )
    )

    # ----------------------------------------------------------------------
    # Figura
    # ----------------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(14, 6)
    )

    posiciones = np.arange(
        1,
        n_grupos + 1
    )

    # ----------------------------------------------------------------------
    # Boxplot
    # ----------------------------------------------------------------------

    caja = ax.boxplot(
        datos_grupos,
        positions=posiciones,
        widths=0.6,
        patch_artist=True,
        showfliers=True,

        medianprops=dict(
            color="black",
            linewidth=1
        ),

        flierprops=dict(
            marker="o",
            markerfacecolor="white",
            markeredgecolor="black",
            markersize=5
        ),
    )

    for parche, color in zip(
        caja["boxes"],
        colores
    ):

        parche.set_facecolor(
            color
        )

        parche.set_alpha(
            0.85
        )

    # ----------------------------------------------------------------------
    # Puntos individuales con jitter
    # ----------------------------------------------------------------------

    rng = np.random.default_rng(
        0
    )

    for pos, valores in zip(
        posiciones,
        datos_grupos
    ):

        jitter = rng.normal(
            0,
            0.05,
            size=len(valores)
        )

        ax.scatter(
            pos + jitter,
            valores,
            color="dimgray",
            alpha=0.6,
            edgecolor="black",
            linewidth=0.3,
            s=18,
            zorder=3
        )

    # ----------------------------------------------------------------------
    # Media
    # ----------------------------------------------------------------------

    for pos, valores in zip(
        posiciones,
        datos_grupos
    ):

        if len(valores) == 0:
            continue

        media = np.mean(
            valores
        )

        ax.scatter(
            pos,
            media,
            color="black",
            edgecolor="white",
            linewidth=0.5,
            s=90,
            zorder=4,
            label=(
                "Sample Mean"
                if pos == posiciones[0]
                else None
            )
        )

        ax.annotate(
            f"{media:.2f}",
            (pos, media),
            textcoords="offset points",
            xytext=(0, 6),
            ha="center",
            fontsize=8,
            color="black",
            fontweight="bold"
        )

    # ----------------------------------------------------------------------
    # Línea de referencia
    # ----------------------------------------------------------------------

    ax.axhline(
        valor_referencia,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=(
            f"Reference "
            f"({valor_referencia:.1f})"
        ),
        zorder=2
    )

    # ----------------------------------------------------------------------
    # Etiquetas
    # ----------------------------------------------------------------------

    ax.set_xticks(
        posiciones
    )

    ax.set_xticklabels(
        etiquetas,
        rotation=20,
        ha="right"
    )

    ax.set_xlabel(
        "Sample"
    )

    ax.set_ylabel(
        "Security SUS Score"
    )

    ax.set_title(
        titulo
    )

    ax.grid(
        axis="y",
        linestyle="-",
        alpha=0.3
    )

    ax.legend(
        loc="lower left"
    )

    fig.tight_layout()

    # ----------------------------------------------------------------------
    # Guardar PNG
    # ----------------------------------------------------------------------

    fig.savefig(
        archivo_salida,
        dpi=150,
        bbox_inches="tight"
    )

    print(
        f"Guardado como {archivo_salida}"
    )

    # ----------------------------------------------------------------------
    # Guardar PDF
    # ----------------------------------------------------------------------

    fig.savefig(
        archivo_salida_pdf,
        bbox_inches="tight"
    )

    print(
        f"Guardado como {archivo_salida_pdf}"
    )

    plt.close(fig)


# --------------------------------------------------------------------------
# MAIN
# --------------------------------------------------------------------------
def main():

    # Cargar y filtrar
    df = cargar_datos(
        ARCHIVO,
        HOJA
    )

    # ----------------------------------------------------------------------
    # Estadísticos
    # ----------------------------------------------------------------------

    resumen = calcular_estadisticos(
        df,
        COL_TRATAMIENTO,
        COL_VALOR,
        GRUPOS_BOXPLOT
    )

    print(
        "\nResumen descriptivo de SUS de seguridad "
        "por tratamiento:\n"
    )

    print(
        resumen.to_string(
            index=False
        )
    )

    # ----------------------------------------------------------------------
    # Exportar Excel
    # ----------------------------------------------------------------------

    resumen.to_excel(
        "resumen_tratamientos_seguridad.xlsx",
        index=False
    )

    print(
        "\nGuardado como "
        "resumen_tratamientos_seguridad.xlsx"
    )

    # ----------------------------------------------------------------------
    # Exportar LaTeX
    # ----------------------------------------------------------------------

    exportar_latex(
        resumen
    )

    # ----------------------------------------------------------------------
    # Graficar
    # ----------------------------------------------------------------------

    graficar_boxplot(
        df,
        COL_TRATAMIENTO,
        COL_VALOR,
        GRUPOS_BOXPLOT,
        TITULO_GRAFICO,
        VALOR_REFERENCIA,
        archivo_salida="boxplot_seguridad_sin_fila_54.png",
        archivo_salida_pdf="boxplot_seguridad_sin_fila_54.pdf"
    )

    # ----------------------------------------------------------------------
    # Información final
    # ----------------------------------------------------------------------

    print(
        "\n" + "=" * 70
    )

    print(
        "ANÁLISIS FINALIZADO"
    )

    print(
        "=" * 70
    )

    print(
        f"\nN final utilizado: {len(df)}"
    )


# --------------------------------------------------------------------------
# EJECUCIÓN
# --------------------------------------------------------------------------
if __name__ == "__main__":
    main()

Participantes originales: 67
Después de excluir quienes no respondieron ningún ítem de seguridad: n=60
Excluyendo participante de la fila Excel 54 (índice pandas 52)
  Grupo: 2
  seguridad sus original: nan
  Motivo: no respondió el primer ítem de seguridad.
Después de excluir fila Excel 54: n=59

Resumen descriptivo de SUS de seguridad por tratamiento:

                     Tratamiento  N  Media  Mediana  Desvío estándar  Mínimo  Máximo
                     T1 - Mobile 14  38.57    36.25            12.85    17.5    60.0
  T2 - Two devices initiating PC 13  37.50    37.50            11.86    10.0    60.0
T3 - 2 devices initiating Mobile 18  43.06    43.75            14.69    15.0    60.0
                         T4 - PC 14  46.25    45.00             7.45    32.5    57.5
       T1 + T4 - One device only 28  42.41    42.50            11.02    17.5    60.0
           T2 + T3 - Two devices 31  40.73    42.50            13.65    10.0    60.0

Guardado como resumen_tratamientos_seguridad.xl

Guardado como boxplot_seguridad_sin_fila_54.pdf

ANÁLISIS FINALIZADO

N final utilizado: 59
